# 01 – Initial data audit (methodology)
## Question
What is the structure and coverage of the six repository snapshots?
Variables: listing ID, city, scrape date, price, minimum nights, bedrooms and instant booking.
Methods: dimensions, duplicate-ID checks, missing-value counts and scrape-date ranges. This preliminary audit supports the eight substantive studies; it is not counted as a separate study.


In [1]:
# Reload src/ automatically when its code changes (no kernel restart needed)
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.cities import CITIES, cities_in_pair
from src.load import load_listings

bcn = load_listings("barcelona")
la = load_listings("los_angeles")

In [2]:
print('Barcelona dimensions (rows, columns):', bcn.shape)
bcn.columns.tolist()


Barcelona dimensions (rows, columns): (15293, 91)


['city',
 'id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_profile_id',
 'host_profile_url',
 'host_name',
 'host_since',
 'hosts_time_as_user_years',
 'hosts_time_as_user_months',
 'hosts_time_as_host_years',
 'hosts_time_as_host_months',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'price_quote_checkin_date',
 'price_quote_checkout_date',
 'price_quote_total_price',
 'pr

In [3]:
# Podgląd typu danych i niepustych wartości dla każdej kolumny
bcn.info(verbose=True, show_counts=True)


<class 'pandas.DataFrame'>
RangeIndex: 15293 entries, 0 to 15292
Data columns (total 91 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   city                                          15293 non-null  str    
 1   id                                            15293 non-null  int64  
 2   listing_url                                   15293 non-null  str    
 3   scrape_id                                     15293 non-null  int64  
 4   last_scraped                                  15293 non-null  str    
 5   source                                        15293 non-null  str    
 6   name                                          15293 non-null  str    
 7   description                                   14644 non-null  str    
 8   neighborhood_overview                         0 non-null      float64
 9   picture_url                                   15293 non-null  str    
 1

## Findings
The audit below documents exact sample counts, scrape dates and missingness. The source symbol `$` does not identify the currency; the city registry supplies local currency. Missing prices are retained in analyses that do not use price. No external data is merged into the listings.


In [4]:
from src.load import load_all
from src.study_utils import export_table
cols = ['id', 'last_scraped', 'price', 'minimum_nights', 'bedrooms', 'instant_bookable']
audit_data = load_all(usecols=cols)
assert not audit_data.duplicated(['city','id']).any()
audit = audit_data.groupby('city').agg(listings=('id','size'),
    scraped_from=('last_scraped','min'), scraped_to=('last_scraped','max'),
    missing_price=('price',lambda s:s.isna().sum()),
    nonpositive_price=('price',lambda s:(s <= 0).sum()),
    missing_min_nights=('minimum_nights',lambda s:s.isna().sum()),
    missing_bedrooms=('bedrooms',lambda s:s.isna().sum()),
    missing_instant_bookable=('instant_bookable',lambda s:s.isna().sum()))
audit['missing_price_pct'] = 100 * audit.missing_price / audit.listings
export_table(audit, 'data_audit', index=True)


,listings,scraped_from,scraped_to,missing_price,nonpositive_price,missing_min_nights,missing_bedrooms,missing_instant_bookable,missing_price_pct
city,,,,,,,,,
barcelona,15293,2026-06-24,2026-07-03,1938,0,1,2562,15293,12.672465
los_angeles,43751,2026-06-15,2026-06-23,5565,0,11,9572,43751,12.719709
madrid,22708,2026-06-20,2026-07-02,3536,0,2,5719,22708,15.571605
mexico_city,31430,2026-06-16,2026-07-06,1793,0,11,5338,31430,5.704741
new_york,30259,2026-06-14,2026-06-23,8744,0,2,10973,30259,28.897188
tokyo,34419,2026-06-30,2026-07-03,2058,0,3,2963,34419,5.979256
